In [5]:
import pandas as pd
import numpy as np

talks = pd.read_csv("/content/ted_talks_cleaned.csv", dtype={"talk_id": str})
interactions = pd.read_csv("/content/user_interactions.csv", dtype={"user_id": str, "talk_id": str})

talks["talk_id"] = talks["talk_id"].astype(str)
interactions["talk_id"] = interactions["talk_id"].astype(str)
interactions["rating"] = pd.to_numeric(interactions["rating"], errors="coerce")

talks = talks.dropna(subset=["talk_id", "title", "combined_text"])
interactions = interactions.dropna(subset=["user_id", "talk_id", "rating"])

In [6]:
user_id = "U001"  # Change this to another user ID

user_history = interactions[interactions["user_id"] == user_id]
print("User's interactions:", len(user_history))
display(user_history.head())

User's interactions: 26


,user_id,talk_id,rating,liked,disliked,watched,saved,interaction_type
0,U001,1795,5,True,False,True,False,like
1,U001,372,4,True,False,True,False,like
2,U001,2327,5,True,False,True,False,like
3,U001,190,5,True,False,True,False,like
4,U001,385,5,True,False,True,True,like


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2
)

talk_vectors = vectorizer.fit_transform(talks["combined_text"].fillna(""))

liked_ids = user_history.loc[user_history["rating"] >= 4, "talk_id"].tolist()
liked_rows = talks.index[talks["talk_id"].isin(liked_ids)].tolist()

if liked_rows:
    # Convert the resulting np.matrix to a standard numpy array to satisfy scikit-learn requirements
    user_profile = np.asarray(talk_vectors[liked_rows].mean(axis=0))
    content_scores = cosine_similarity(user_profile, talk_vectors).flatten()

    content_recs = talks[["talk_id", "title"]].copy()
    content_recs["content_score"] = content_scores
    content_recs = content_recs[~content_recs["talk_id"].isin(user_history["talk_id"])]
    content_recs = content_recs.sort_values("content_score", ascending=False).head(20)
else:
    content_recs = pd.DataFrame(columns=["talk_id", "title", "content_score"])
    print("No highly rated talks found for this user.")

display(content_recs.head(10))

,talk_id,title,content_score
1652,1953,Where's Google going next?,0.480948
1861,2182,How to run a company with (almost) no rules,0.457790
99,118,The genesis of Google,0.454017
3973,61930,It's OK to feel overwhelmed. Here's what to do...,0.453534
363,436,Design and discovery,0.452245
353,426,Play! Experiment! Discover!,0.448474
332,396,Fashion and creativity,0.438485
3982,62266,What coronavirus means for the global economy,0.430550
78,74,The route to a sustainable future,0.425179
1302,1571,"How art, technology and design inform creative...",0.423163


In [10]:
user_item = interactions.pivot_table(
    index="user_id",
    columns="talk_id",
    values="rating",
    aggfunc="mean"
).fillna(0)

user_similarity = cosine_similarity(user_item)
user_similarity = pd.DataFrame(
    user_similarity,
    index=user_item.index,
    columns=user_item.index
)

if user_id in user_similarity.index:
    # Get similarities, and set the target user's self-similarity to 0 to keep matrices aligned
    similar_users = user_similarity[user_id].copy()
    similar_users[user_id] = 0
    similar_users = similar_users.sort_values(ascending=False)

    weighted_scores = user_item.T.dot(similar_users)
    similarity_totals = user_item.ne(0).T.dot(similar_users)

    predicted_scores = weighted_scores / similarity_totals.replace(0, np.nan)
    predicted_scores = predicted_scores.dropna()

    already_rated = user_history["talk_id"].tolist()
    predicted_scores = predicted_scores.drop(index=already_rated, errors="ignore")

    collaborative_recs = predicted_scores.rename("collaborative_score").reset_index()
    collaborative_recs = collaborative_recs.rename(columns={"talk_id": "talk_id"})
    collaborative_recs = collaborative_recs.merge(
        talks[["talk_id", "title"]].drop_duplicates("talk_id"),
        on="talk_id",
        how="left"
    )
    collaborative_recs = collaborative_recs.sort_values(
        "collaborative_score",
        ascending=False
    ).head(20)
else:
    collaborative_recs = pd.DataFrame(
        columns=["talk_id", "collaborative_score", "title"]
    )
    print("User ID was not found in the interaction data.")

display(collaborative_recs.head(10))

,talk_id,collaborative_score,title
67,42248,4.929781,"""The Nutritionist"""
4,1267,4.923386,A map of the brain
14,15518,4.8764,Why you should make useless things
86,79,4.771862,How mobile phones can fight poverty
22,1826,4.725606,How reliable is your memory?
6,13029,4.663093,"The ""dead zone"" of the Gulf of Mexico"
40,2265,4.650876,The surprisingly logical minds of babies
7,1307,4.638836,Medicine for the 99 percent
9,1336,4.570683,Extreme swimming with the world's most dangero...
23,1879,4.568419,What is so special about the human brain?


In [11]:
content_weight = 0.5  # 0 = collaborative only; 1 = content only

def min_max_normalize(series):
    if series.empty or series.max() == series.min():
        return series * 0
    return (series - series.min()) / (series.max() - series.min())

content_part = content_recs[["talk_id", "title", "content_score"]].copy()
collab_part = collaborative_recs[["talk_id", "title", "collaborative_score"]].copy()

content_part["content_score"] = min_max_normalize(content_part["content_score"])
collab_part["collaborative_score"] = min_max_normalize(collab_part["collaborative_score"])

hybrid_recs = content_part.merge(
    collab_part[["talk_id", "collaborative_score"]],
    on="talk_id",
    how="outer"
)

hybrid_recs["content_score"] = hybrid_recs["content_score"].fillna(0)
hybrid_recs["collaborative_score"] = hybrid_recs["collaborative_score"].fillna(0)

hybrid_recs["hybrid_score"] = (
    content_weight * hybrid_recs["content_score"]
    + (1 - content_weight) * hybrid_recs["collaborative_score"]
)

hybrid_recs = hybrid_recs.drop_duplicates(subset="talk_id")
hybrid_recs = hybrid_recs[
    ~hybrid_recs["talk_id"].isin(user_history["talk_id"])
]
hybrid_recs = hybrid_recs.sort_values("hybrid_score", ascending=False)

display(hybrid_recs.head(10))

/tmp/ipykernel_393/4043429484.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  hybrid_recs["collaborative_score"] = hybrid_recs["collaborative_score"].fillna(0)


,talk_id,title,content_score,collaborative_score,hybrid_score
26,42248,NaN,0.000000,1.000000,0.500000
13,1953,Where's Google going next?,1.000000,0.000000,0.500000
1,1267,NaN,0.000000,0.988194,0.494097
6,15518,NaN,0.000000,0.901450,0.450725
15,2182,How to run a company with (almost) no rules,0.720816,0.000000,0.360408
38,79,NaN,0.000000,0.708455,0.354228
0,118,The genesis of Google,0.675329,0.000000,0.337664
33,61930,It's OK to feel overwhelmed. Here's what to do...,0.669504,0.000000,0.334752
29,436,Design and discovery,0.653957,0.000000,0.326979
9,1826,NaN,0.000000,0.623058,0.311529


In [12]:
content_recs["model"] = "Content"
collaborative_recs["model"] = "Collaborative"
hybrid_recs["model"] = "Hybrid"

comparison = pd.concat([
    content_recs[["model", "talk_id", "title"]].head(10),
    collaborative_recs[["model", "talk_id", "title"]].head(10),
    hybrid_recs[["model", "talk_id", "title"]].head(10)
], ignore_index=True)

display(comparison)

,model,talk_id,title
0,Content,1953,Where's Google going next?
1,Content,2182,How to run a company with (almost) no rules
2,Content,118,The genesis of Google
3,Content,61930,It's OK to feel overwhelmed. Here's what to do...
4,Content,436,Design and discovery
5,Content,426,Play! Experiment! Discover!
6,Content,396,Fashion and creativity
7,Content,62266,What coronavirus means for the global economy
8,Content,74,The route to a sustainable future
9,Content,1571,"How art, technology and design inform creative..."
